In [7]:
import duckdb
import pandas as pd
import requests

In [8]:
# get parquet files from Deadlock API
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def urls(table: str, limit: int = 50) -> list[str]:
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
        if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    return file_list[-limit:]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")

file_urls = urls("match_player", limit=30)
con.execute(f"CREATE VIEW match_player AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

In [9]:
pd.set_option("display.max_colwidth", None)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)
pd.set_option("display.expand_frame_repr", False)

In [24]:
# account_id and start_time are used for the dashboard
extracted_data = con.execute("""
WITH valid_matches AS (
    SELECT match_id
    FROM match_player
    GROUP BY match_id
    HAVING COUNT(*) = 12
),

filter_table AS (
    SELECT *
    FROM match_player
    WHERE match_id IN (SELECT match_id FROM valid_matches) 
    AND game_mode NOT IN ('StreetBrawl', 'Training', 'Tutorial')
    AND LEN("stats.time_stamp_s") > 0
    AND match_outcome != 'Error'
)
SELECT match_id, player_slot, team, won,
account_id, start_time, hero_id,
COLUMNS('^stats\\.'), game_mode
FROM filter_table
LIMIT 360
""").df()
#LIMIT is to be removed when using for df

player_timelines = []

for _, row in extracted_data.iterrows():
    
    # gathers timestamps from the stats column
    timestamps = row["stats.time_stamp_s"]
    
    if timestamps is None:
        continue
    
    # gather static information from the matches
    timeline = pd.DataFrame({"timestamp": timestamps})
    timeline["match_id"] = row["match_id"]
    timeline["team"] = row["team"]
    timeline["won"] = row["won"]
    timeline["start_time"] = row["start_time"]
    timeline["account_id"] = row["account_id"]
    timeline["hero_id"] = row["hero_id"]
    
    
    for col in extracted_data.columns:
        
        # gathers the stats at the given timestamp for the match
        if col.startswith("stats.") and col != "stats.time_stamp_s":
            values = row[col]
            
            if values is not None and len(values) == len(timestamps):
                timeline[col.replace("stats.", "")] = values
                
    player_timelines.append(timeline)


all_timelines = pd.concat(player_timelines, ignore_index=True)
all_timelines["minutes"] = all_timelines["timestamp"] / 60
all_timelines.head(25)

# this df can be exported to be used as data for modeling, it should have all info associated with timestamps 

,timestamp,match_id,team,won,start_time,account_id,hero_id,net_worth,gold_player,gold_player_orbs,gold_lane_creep_orbs,gold_neutral_creep_orbs,gold_boss,gold_boss_orb,gold_treasure,gold_denied,gold_death_loss,gold_lane_creep,gold_neutral_creep,kills,deaths,assists,creep_kills,neutral_kills,possible_creeps,creep_damage,player_damage,neutral_damage,boss_damage,denies,player_healing,ability_points,self_healing,player_barriering,teammate_healing,teammate_barriering,self_damage,bullet_kills,melee_kills,ability_kills,headshot_kills,player_damage_taken,max_health,weapon_power,tech_power,shots_hit,shots_missed,damage_absorbed,absorption_provided,hero_bullets_hit,hero_bullets_hit_crit,heal_prevented,heal_lost,damage_mitigated,level,gold_source_assists_gold,gold_source_team_bonus_gold,gold_source_breakable_gold,gold_source_ability_assassinate_gold,gold_source_item_trophy_collector_gold,gold_source_item_cultist_sacrifice_gold,gold_source_item_goose_egg_gold,minutes
0,180,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,1608,0,0,527,0,0,0,0,0,0,481,0,0,0,0,8,0,16,2178,719,0,0,0,0,2,0,0,0,0,0,0,0,0,0,441,881,0,3,76,18,0,0,21,8,0,0,0,4,0,0,0,0,0,0,0,3.000000
1,360,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,3324,20,0,1355,0,0,0,0,0,0,1313,0,0,0,0,18,0,36,5739,1433,0,0,0,131,4,131,0,0,0,0,0,0,0,0,1418,1087,9,13,150,26,0,0,36,8,0,0,210,7,0,4,32,0,0,0,0,6.000000
2,540,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,6314,615,0,2109,0,0,0,0,0,0,2521,0,0,0,0,28,0,56,10140,2694,0,0,0,308,7,308,0,0,0,0,0,0,0,0,3539,1411,9,22,201,49,0,0,51,11,0,0,878,11,0,4,425,0,0,0,0,9.000000
3,720,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,9999,658,0,3609,0,394,0,0,0,0,4199,0,0,0,0,41,0,80,15326,3233,0,0,0,364,12,364,0,0,0,0,0,0,0,0,5101,1897,9,54,239,56,0,0,57,12,0,0,1607,16,0,4,425,0,0,0,0,12.000000
4,900,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,12746,658,0,4172,0,394,0,278,21,0,5216,162,0,0,0,49,3,92,18731,4887,633,0,0,510,15,510,0,0,0,0,0,0,0,0,5941,2042,9,57,259,63,0,0,62,13,0,0,1758,19,0,4,1126,0,0,0,0,15.000000
5,1200,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,16853,1193,0,4909,0,932,0,278,21,0,5951,603,1,0,1,53,9,100,21047,6457,2826,341,0,581,19,581,0,0,0,0,0,0,1,0,6145,2622,13,76,281,81,0,0,70,15,0,0,2309,23,138,4,2162,0,0,0,0,20.000000
6,1500,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,24285,2021,0,6442,0,2337,0,930,21,0,8121,603,1,0,6,67,9,124,27081,10553,2826,532,0,996,23,996,0,0,0,0,0,0,1,0,8582,2873,13,93,321,102,0,0,88,18,0,0,4424,27,881,4,2821,0,0,0,0,25.000000
7,1800,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,31280,3322,0,8066,0,2801,0,930,21,0,11369,603,1,0,8,85,9,152,36243,15775,2826,532,0,1295,26,1295,0,0,0,0,0,0,1,0,12615,3047,13,112,348,124,0,0,97,19,0,0,6662,30,2052,4,3119,0,0,0,0,30.000000
8,2100,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,39299,3455,0,10277,0,3217,0,1880,21,0,14618,1290,1,1,9,110,12,187,46177,19778,8153,532,0,2070,29,2070,0,0,0,0,0,0,1,0,16914,3261,19,116,403,139,0,0,106,22,0,0,10238,33,2174,4,3457,0,0,0,0,35.000000
9,2400,134006693,Team1,False,2026-11-08 19:08:55-05:00,80371642,69,46911,4483,0,12598,0,3592,0,1880,21,0,17009,2549,2,2,12,122,12,201,54669,24666,8153,532,0,3311,31,3311,0,0,0,0,0,0,2,0,23780,3407,19,147,428,161,0,0,121,24,0,157,13972,35,3192,4,3635,0,0,0,0,40.000000


In [25]:
for col in all_timelines.columns:
    print(col)

timestamp
match_id
team
won
start_time
account_id
hero_id
net_worth
gold_player
gold_player_orbs
gold_lane_creep_orbs
gold_neutral_creep_orbs
gold_boss
gold_boss_orb
gold_treasure
gold_denied
gold_death_loss
gold_lane_creep
gold_neutral_creep
kills
deaths
assists
creep_kills
neutral_kills
possible_creeps
creep_damage
player_damage
neutral_damage
boss_damage
denies
player_healing
ability_points
self_healing
player_barriering
teammate_healing
teammate_barriering
self_damage
bullet_kills
melee_kills
ability_kills
headshot_kills
player_damage_taken
max_health
weapon_power
tech_power
shots_hit
shots_missed
damage_absorbed
absorption_provided
hero_bullets_hit
hero_bullets_hit_crit
heal_prevented
heal_lost
damage_mitigated
level
gold_source_assists_gold
gold_source_team_bonus_gold
gold_source_breakable_gold
gold_source_ability_assassinate_gold
gold_source_item_trophy_collector_gold
gold_source_item_cultist_sacrifice_gold
gold_source_item_goose_egg_gold
minutes


In [ ]:
# exclude all columns not representing stats

### not needed if we are not averaging columns by team
stat_columns = [
    col for col in all_timelines.columns
    if col not in [
        "timestamp",
        "match_id",
        "team",
        "won",
        "minutes"
    ]
]

# create a simple variable for each stats column to more easily use SQL 
avg_columns = ", ".join(
    f'AVG("{col}") AS "{col}"' 
    for col in stat_columns
    )


team_timelines = con.execute(f"""
SELECT match_id,
team,
won,
timestamp,
minutes,
{avg_columns}
FROM all_timelines
GROUP BY match_id, team, won, timestamp, minutes
ORDER BY match_id, team, timestamp, minutes
""").df()

team_timelines.head(25)

,match_id,team,won,timestamp,minutes,net_worth,gold_player,gold_player_orbs,gold_lane_creep_orbs,gold_neutral_creep_orbs,gold_boss,gold_boss_orb,gold_treasure,gold_denied,gold_death_loss,gold_lane_creep,gold_neutral_creep,kills,deaths,assists,creep_kills,neutral_kills,possible_creeps,creep_damage,player_damage,neutral_damage,boss_damage,denies,player_healing,ability_points,self_healing,player_barriering,teammate_healing,teammate_barriering,self_damage,bullet_kills,melee_kills,ability_kills,headshot_kills,player_damage_taken,max_health,weapon_power,tech_power,shots_hit,shots_missed,damage_absorbed,absorption_provided,hero_bullets_hit,hero_bullets_hit_crit,heal_prevented,heal_lost,damage_mitigated,level,gold_source_assists_gold,gold_source_team_bonus_gold,gold_source_breakable_gold,gold_source_ability_assassinate_gold,gold_source_item_trophy_collector_gold,gold_source_item_cultist_sacrifice_gold,gold_source_item_goose_egg_gold
0,134547479,Team0,False,180,3.000000,1771.000000,71.000000,0.0,353.333333,0.0,0.000000,0.0,0.000000,51.166667,0.000000,650.500000,45.000000,0.166667,0.166667,0.166667,7.000000,1.000000,10.166667,1844.666667,1125.833333,190.166667,72.166667,1.000000,128.500000,1.833333,128.500000,0.000000,0.000000,0.000000,13.333333,0.000000,0.166667,0.000000,0.000000,1009.666667,939.666667,8.333333,7.000000,192.333333,141.500000,0.000000,0.000000,44.166667,8.833333,0.000000,0.000000,207.166667,4.000000,22.166667,0.000000,0.000000,0.0,0.000000,0.0,0.0
1,134547479,Team0,False,360,6.000000,3586.333333,179.666667,0.0,905.500000,0.0,0.000000,0.0,0.000000,77.666667,0.000000,1425.000000,138.000000,0.333333,0.166667,0.333333,14.833333,3.000000,21.833333,4078.166667,2689.833333,564.666667,214.666667,1.500000,470.833333,3.833333,448.333333,53.833333,22.500000,0.000000,67.500000,0.000000,0.166667,0.166667,0.000000,2197.000000,1104.333333,11.333333,17.666667,391.833333,271.166667,53.833333,53.833333,84.333333,16.833333,0.000000,0.000000,449.333333,7.166667,56.000000,0.000000,260.500000,0.0,0.000000,0.0,0.0
2,134547479,Team0,False,540,9.000000,6039.166667,798.500000,0.0,1452.833333,0.0,0.000000,0.0,0.000000,77.666667,37.833333,2117.833333,343.666667,1.000000,0.833333,1.500000,21.666667,4.833333,33.333333,6304.000000,4595.833333,1111.166667,271.166667,1.500000,978.833333,6.833333,954.166667,116.000000,24.666667,0.000000,125.166667,0.333333,0.166667,0.500000,0.000000,4049.666667,1302.833333,11.333333,40.500000,554.833333,382.000000,116.000000,116.000000,119.166667,21.000000,0.000000,0.000000,747.500000,10.833333,283.166667,74.666667,598.500000,0.0,0.000000,0.0,0.0
3,134547479,Team0,False,720,12.000000,9120.000000,1403.833333,0.0,2050.333333,0.0,229.000000,0.0,0.000000,87.666667,67.333333,2910.666667,712.500000,1.666667,1.333333,2.166667,30.500000,6.666667,45.000000,9061.333333,7099.833333,1748.500000,1504.000000,1.666667,1381.166667,10.500000,1354.000000,147.666667,27.166667,0.000000,213.833333,0.666667,0.333333,0.666667,0.166667,6164.666667,1629.833333,22.666667,54.833333,700.166667,474.500000,147.666667,147.666667,146.166667,27.833333,47.333333,0.000000,1140.666667,14.500000,458.500000,255.000000,895.000000,0.0,0.000000,0.0,0.0
4,134547479,Team0,False,900,15.000000,12823.500000,2204.666667,0.0,2567.000000,0.0,687.000000,0.0,0.000000,87.666667,309.666667,3615.833333,1589.666667,2.666667,1.833333,4.166667,38.000000,12.000000,55.333333,12631.333333,9582.000000,3751.000000,1701.000000,1.666667,2035.166667,14.833333,1934.666667,283.333333,100.500000,0.000000,356.333333,0.833333,0.833333,1.000000,0.166667,7862.666667,2015.500000,21.000000,67.166667,807.833333,562.166667,283.333333,283.333333,167.166667,32.500000,91.833333,0.000000,1932.833333,18.833333,811.333333,415.166667,1306.166667,0.0,0.000000,0.0,0.0
5,134547479,Team0,False,1200,20.000000,18054.666667,2898.500000,0.0,3567.166667,0.0,1299.000000,0.0,432.666667,98.833333,663.166667,4887.166667,2238.833333,3.333333,3.000000,5.333333,51.666667,18.833333,72.500000,17878.333333,13737.500000,